In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 68.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.


In [6]:
import wfdb #library used to read MIT-BIH database
import numpy as np
from scipy.signal import butter , filtfilt
import matplotlib.pyplot as plt
import os

In [7]:


file_path = '/content/drive/My Drive/Evolutionary_ECG_Project/data/100'
record = wfdb.rdrecord(file_path)
raw_signal = record.p_signal[:,0]

def butter_bandpass_filter(data, lowcut, highcut, fs, order = 5):
  nyquist = 0.5*fs #following nyquist criterion
  low = lowcut/nyquist#(for heart low frequency is 0.5 hz)
  high = highcut/nyquist#(for heart high frequency is 50 hz)
  b,a = butter(order, [low,high], btype = 'band')
  y = filtfilt(b,a,data) #applies the filter twice once forward and once backward this would cancel out any delay filter introduces
  return y

clean_signal = butter_bandpass_filter(raw_signal,0.5,50,360,5)

In [10]:

project_path = "/content/drive/My Drive/Evolutionary_ECG_Project"
input_folder = f"{project_path}/data"
output_folder = f"{project_path}/processed_data"
os.makedirs(output_folder, exist_ok=True)

all_files = os.listdir(input_folder)
patient_ids = sorted(list(set([f.split('.')[0] for f in all_files if f.endswith('.dat')])))

for patient in patient_ids:
    file_path = f"{input_folder}/{patient}"
    record = wfdb.rdrecord(file_path)
    raw_signal = record.p_signal[:, 0]
    clean_signal = butter_bandpass_filter(raw_signal, 0.5, 50, 360, 5)
    np.save(f"{output_folder}/{patient}_clean.npy", clean_signal)